# 01. Exploratory Data Analysis (EDA) - Employee Attrition Intelligence

**Project:** AttriGuard - Employee Attrition Prediction System  
**Dataset:** IBM HR Analytics Employee Attrition & Performance  
**Objective:** Uncover key drivers of workforce attrition, analyze class imbalance, audit features for zero-variance and data leakage, and establish empirical foundations for feature engineering and model training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visual style setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette(['#3B82F6', '#EF4444', '#10B981', '#F59E0B'])
pd.set_option('display.max_columns', 40)

## 1. Data Ingestion & Shape Auditing

In [2]:
df = pd.read_csv('../data/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Missing Values, Duplicates, and Zero-Variance Columns Auditing

In [3]:
# Missing value check
missing_counts = df.isnull().sum()
print("Columns with missing values:")
print(missing_counts[missing_counts > 0])

# Duplicate check
print(f"\nDuplicate rows found: {df.duplicated().sum()}")

# Constant (zero-variance) columns check
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print(f"\nZero-variance constant columns to drop: {constant_cols}")

# High-cardinality ID columns
print("Surrogate ID columns: ['EmployeeNumber']")

## 3. Target Distribution: Attrition & Class Imbalance Analysis

Understanding the prior class distribution is crucial. If attrition is rare (~16%), evaluating simply on accuracy is flawed, as a dummy classifier predicting 'No' achieves ~84% accuracy while catching 0% of leaving employees.

In [4]:
attr_counts = df['Attrition'].value_counts()
attr_pct = df['Attrition'].value_counts(normalize=True) * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=attr_counts.index, y=attr_counts.values, ax=ax[0], palette=['#10B981', '#EF4444'])
ax[0].set_title('Attrition Headcount Distribution')
ax[0].set_ylabel('Employee Count')

ax[1].pie(attr_counts.values, labels=attr_counts.index, autopct='%1.1f%%', colors=['#10B981', '#EF4444'], startangle=90)
ax[1].set_title('Attrition Class Percentage')
plt.tight_layout()
plt.show()

print(f"Class Imbalance Ratio: No={attr_pct['No']:.1f}% vs Yes={attr_pct['Yes']:.1f}%")

## 4. Key Attrition Drivers & Bivariate Analyses

In [5]:
# Overtime vs Attrition
plt.figure(figsize=(8, 4))
ot_attr = pd.crosstab(df['OverTime'], df['Attrition'], normalize='index') * 100
ot_attr.plot(kind='bar', stacked=True, color=['#10B981', '#EF4444'], figsize=(8, 4))
plt.title('Attrition Rate by Overtime Work Status (%)')
plt.ylabel('Percentage')
plt.xlabel('OverTime')
plt.legend(title='Attrition')
plt.show()

In [6]:
# Monthly Income distribution by Attrition
plt.figure(figsize=(10, 4))
sns.kdeplot(data=df, x='MonthlyIncome', hue='Attrition', common_norm=False, fill=True, palette=['#10B981', '#EF4444'])
plt.title('Monthly Income Density: Attrition Yes vs No')
plt.xlabel('Monthly Income ($)')
plt.show()

In [7]:
# Job Role vs Attrition Rate
role_attr = pd.crosstab(df['JobRole'], df['Attrition'], normalize='index')['Yes'] * 100
role_attr_sorted = role_attr.sort_values(ascending=False)

plt.figure(figsize=(12, 5))
sns.barplot(x=role_attr_sorted.values, y=role_attr_sorted.index, palette='flare')
plt.title('Attrition Rate by Job Role (%)')
plt.xlabel('Attrition Rate (%)')
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap of Numerical Features

In [8]:
num_cols = df.select_dtypes(include=[np.number]).columns.drop(['EmployeeCount', 'StandardHours', 'EmployeeNumber'], errors='ignore')
corr = df[num_cols].corr()

plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap='vlag', center=0, annot=False, linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## 6. Summary of Key EDA Findings

1. **Severe Imbalance:** Only ~16.1% attrition rate. Strategy: Use stratified train/test splitting, class-balanced weights (`class_weight='balanced'`), and evaluate using ROC-AUC, Recall, and Precision-Recall curves rather than raw accuracy.
2. **OverTime Impact:** Employees working overtime experience more than double the attrition rate (~30% vs ~10%).
3. **Compensation Threshold:** Sharp attrition density under $3,500/month; substantial stabilization above $8,000/month.
4. **Role Sensitivity:** Sales Representatives and Laboratory Technicians demonstrate the highest departure rates.
5. **Cleaned Columns:** Dropped zero-variance fields (`EmployeeCount`, `StandardHours`, `Over18`) and arbitrary identifier `EmployeeNumber`.